We start by making a connection with Airport database.

In [3]:
import mysql.connector
from mysql.connector import Error
import prettytable

#Creating a connection with Employees database

def create_connection(host_name, user_name, user_password):
    connection = None
    try:
        connection = mysql.connector.connect(
            host=host_name,
            user=user_name,
            passwd=user_password,
            database='airportdb'
        )
        print("Connection to MySQL DB successful")
    except Error as e:
        print(f"The error '{e}' occurred")

    return connection

connection = create_connection("localhost", "root", "kurssql11")
my_cursor=connection.cursor()  


Connection to MySQL DB successful


We start by finding the total revenue from flight tickets.

In [2]:
my_cursor.execute("""
SELECT
	CONCAT(ROUND(SUM(price)/1000000,2),' mln') as total_revenue
FROM booking
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+---------------+
| total_revenue |
+---------------+
|  13629.59 mln |
+---------------+


Now we are intrested in monthly revenue distribution.

In [3]:
my_cursor.execute("""
WITH monthly_revenue AS
			(
			SELECT
				DATE_FORMAT(f.departure, "%Y-%m-01") AS month,
				SUM(b.price) AS revenue
			FROM flight f
			JOIN booking b
				ON b.flight_id=f.flight_id
			GROUP BY month
			ORDER BY month
			)

SELECT 
	month,
    revenue,
	SUM(revenue) OVER (ORDER BY month) AS running_revenue
FROM monthly_revenue
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+------------+---------------+-----------------+
|   month    |    revenue    | running_revenue |
+------------+---------------+-----------------+
| 2015-06-01 | 4402788741.38 |  4402788741.38  |
| 2015-07-01 | 4535061010.08 |  8937849751.46  |
| 2015-08-01 | 4539969453.11 |  13477819204.57 |
| 2015-09-01 |  151771352.61 |  13629590557.18 |
+------------+---------------+-----------------+


Now we are interested in average, minimal and maximal ticket price.

In [3]:
my_cursor.execute("""
SELECT
	ROUND(AVG(price),2) AS average_price,
    MIN(price) AS minimal_price,
    MAX(price) AS maximal_price
FROM booking
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+---------------+---------------+---------------+
| average_price | minimal_price | maximal_price |
+---------------+---------------+---------------+
|     250.98    |      1.00     |     501.00    |
+---------------+---------------+---------------+


After preliminary analysis we are interested in distribution of ticket prices.

In [4]:
my_cursor.execute("""
SELECT
    CONCAT(ROUND(price/50)*50,'-',ROUND(price/50)*50+50) AS price_interval,
    CONCAT(ROUND(COUNT(booking_id)/1000000,4),' mln') as frequency
FROM booking
GROUP BY price_interval
ORDER BY price_interval
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+----------------+------------+
| price_interval | frequency  |
+----------------+------------+
|      0-50      | 2.6085 mln |
|    100-150     | 5.4340 mln |
|    150-200     | 5.4275 mln |
|    200-250     | 5.4348 mln |
|    250-300     | 5.4389 mln |
|    300-350     | 5.4352 mln |
|    350-400     | 5.4367 mln |
|    400-450     | 5.4286 mln |
|    450-500     | 5.4241 mln |
|     50-100     | 5.4186 mln |
|    500-550     | 2.8177 mln |
+----------------+------------+


Next we find most profitable flights. Using the following query, we get our answer but the execution time is long (1421sec=24min). This is mainly because we join tables to booking table which has aprox. 54mln rows. To optimize we should first find just flight ids with the biggest revenue and then join tables to get more information about flights.

In [ ]:
"""SELECT
	b.flight_id,
	a1.city as departure_city,
    a1.country as departure_country,
    a2.city as destination_city,
    a2.country as destination_country,
    SUM(b.price) as revenue
FROM booking b
LEFT JOIN flight f
	ON f.flight_id=b.flight_id
LEFT JOIN  airport_geo a1
	on a1.airport_id=f.from
LEFT JOIN  airport_geo a2
	on a2.airport_id=f.to
GROUP BY b.flight_id, a1.city, a1.country, a2.city, a2.country
ORDER BY revenue DESC
LIMIT 10;
"""

In [5]:
my_cursor.execute("""
WITH revenue AS
			(
			SELECT
				flight_id,
				SUM(price) as revenue
			FROM booking
			GROUP BY flight_id
			ORDER BY revenue DESC
			LIMIT 10
			)
            
SELECT
	r.flight_id,
    a1.city as departure_city,
    a1.country as departure_country,
    a2.city as destination_city,
    a2.country as destination_country,
    r.revenue
FROM revenue r
LEFT JOIN flight f
	ON f.flight_id=r.flight_id
LEFT JOIN  airport_geo a1
	on a1.airport_id=f.from
LEFT JOIN  airport_geo a2
	on a2.airport_id=f.to;
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+-----------+-----------------+-------------------+------------------+---------------------+----------+
| flight_id |  departure_city | departure_country | destination_city | destination_country | revenue  |
+-----------+-----------------+-------------------+------------------+---------------------+----------+
|   396362  |    KARLSHOFEN   |      GERMANY      |   CUMBERNAULD    |    UNITED KINGDOM   | 97937.80 |
|   320213  |      MEHAMN     |       NORWAY      |      GEMENA      |        CONGO        | 96440.82 |
|   93402   |     BLANDING    |   UNITED STATES   |     WUNSTORF     |       GERMANY       | 96075.27 |
|   77871   |      SERRO      |       BRAZIL      |   MEADOW LAKE    |        CANADA       | 95621.95 |
|   615019  |      URRAO      |      COLOMBIA     |     BRUCHSAL     |       GERMANY       | 95408.76 |
|   590361  |     HAMBURG     |      GERMANY      |    NYUTABARU     |        JAPAN        | 95262.47 |
|   45470   | BANDAR MAHSHAHR |        IRAN       |     AUGUSTA 

After opitmization query took only 507 second, so we reduced the execution time aprox. by 64%.

Now we look for the airlines that made the highest revenue.

In [4]:
my_cursor.execute("""
WITH revenue AS
			(
			SELECT
				flight_id,
				SUM(price) as revenue
			FROM booking
			GROUP BY flight_id
			ORDER BY revenue DESC
			LIMIT 10
			),

total_revenue AS
			(	
			SELECT 
				f.airline_id,
				SUM(r.revenue) AS total_revenue
			FROM revenue r
			LEFT JOIN flight f
				ON f.flight_id=r.flight_id
			GROUP BY f.airline_id
			)
            
SELECT
	a.airlinename,
    tr.total_revenue
FROM total_revenue tr
LEFT JOIN airline a
	ON a.airline_id=tr.airline_id
""")

my_result=my_cursor.fetchall() #Uploading data
headers = [i[0] for i in my_cursor.description] #Uploading headers
x = prettytable.PrettyTable(headers)

for v in my_result:
    x.add_row(v)

print(x)

+---------------------+---------------+
|     airlinename     | total_revenue |
+---------------------+---------------+
|    Oman Airlines    |    97937.80   |
|   Croatia Airlines  |    96440.82   |
|    Peru Airlines    |    96075.27   |
|  Thailand Airlines  |    95621.95   |
|  Colombia Airlines  |    95408.76   |
|  Caicos Is Airlines |    95262.47   |
| Uzbekistan Airlines |    95174.75   |
|   Rwanda Airlines   |    95112.52   |
|  Trinidad Airlines  |    94924.90   |
|   Liberia Airlines  |    94899.12   |
+---------------------+---------------+
